# 01 – Łączenie i czyszczenie danych meczowych

Wejście: 100 plików CSV z [football-data.co.uk](https://www.football-data.co.uk/) (5 lig × 20 sezonów, 2005/06–2024/25) oraz ręcznie przygotowany słownik stadionów `teams_stadiums.csv`.

Wyjście: `data/processed/matches_clean.csv` – jeden wiersz na mecz, z lokalizacją stadionu, datą i godziną rozpoczęcia.

In [1]:
import glob
import os
from pathlib import Path

import numpy as np
import pandas as pd

RAW_DIR = Path('../data/raw/football-data')
STADIUMS_FILE = Path('../data/reference/teams_stadiums.csv')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

LEAGUES = {
    'D1': 'Bundesliga',
    'SP1': 'La Liga',
    'E0': 'Premier League',
    'I1': 'Serie A',
    'F1': 'Ligue 1',
}

## Wczytanie i połączenie plików źródłowych

Nazwa pliku ma format `<kod_ligi>_<sezon>.csv`, np. `E0_2019-20.csv`.

In [2]:
all_files = sorted(glob.glob(str(RAW_DIR / '*.csv')))
print(f'Znaleziono {len(all_files)} plików CSV do przetworzenia.')

list_of_dfs = []
for file in all_files:
    league_code, season = Path(file).stem.split('_')
    df_file = pd.read_csv(file, encoding='latin1', low_memory=False)

    # Część plików zaczyna się od znacznika BOM, przez co kolumna 'Div' wczytuje się jako 'ï»¿Div'
    df_file.columns = [col.replace('ï»¿', '') for col in df_file.columns]

    df_file['Division'] = LEAGUES[league_code]
    df_file['Season'] = season
    list_of_dfs.append(df_file)

combined_df = pd.concat(list_of_dfs, ignore_index=True)

# Usuwam puste wiersze na końcach niektórych plików
combined_df = combined_df.dropna(subset=['HomeTeam'])

print(f'Wymiary połączonego zbioru (wiersze, kolumny): {combined_df.shape}')

Znaleziono 100 plików CSV do przetworzenia.


Wymiary połączonego zbioru (wiersze, kolumny): (36271, 170)


In [3]:
combined_df['Division'].value_counts()

Division
Premier League    7600
Serie A           7600
La Liga           7600
Ligue 1           7351
Bundesliga        6120
Name: count, dtype: int64

## Dołączenie lokalizacji stadionów

In [4]:
df_teams_stadiums = pd.read_csv(STADIUMS_FILE)
df_teams_stadiums.head()

,Team,City,Stadium,Latitude,Longitude
0,Aachen,Aachen,New Tivoli,50.7934,6.0975
1,Ajaccio,Ajaccio,Stade François Coty,41.9308,8.7765
2,Ajaccio GFCO,Ajaccio,Stade Ange Casanova,41.9514,8.7720
3,Alaves,Vitoria-Gasteiz,Mendizorrotza Stadium,42.8371,-2.6885
4,Almeria,Almería,Power Horse Stadium,36.8400,-2.4354


In [5]:
# Sprawdzam, czy każda drużyna grająca u siebie ma przypisany stadion
missing_teams = set(combined_df['HomeTeam']) - set(df_teams_stadiums['Team'])
print(f'Drużyny bez przypisanego stadionu: {sorted(missing_teams)}')

Drużyny bez przypisanego stadionu: []


In [6]:
matches = pd.merge(combined_df, df_teams_stadiums, left_on='HomeTeam', right_on='Team', how='left')
matches[['Latitude', 'Longitude']].isnull().sum()

Latitude     0
Longitude    0
dtype: int64

## Ujednolicenie dat

Pliki źródłowe mają dwa formaty dat: `31/12/04` (starsze sezony) i `31/12/2024` (nowsze). Każdy wiersz parsuję jawnie jednym z tych formatów, zawsze w kolejności dzień/miesiąc.

> W pierwotnej wersji pracy daty z dniem ≤ 12 zostały zinterpretowane jako miesiąc/dzień (np. 10.08.2007 → 2007-10-08). Błąd dotyczył ok. 35% meczów, więc dane pogodowe dla nich pobrano z niewłaściwego dnia. Szczegóły w README, w sekcji *Known limitations*.

In [7]:
dates_short = pd.to_datetime(matches['Date'], format='%d/%m/%y', errors='coerce')
dates_long = pd.to_datetime(matches['Date'], format='%d/%m/%Y', errors='coerce')
matches['Date'] = dates_short.fillna(dates_long)

print(f"Nieprzetworzone daty: {matches['Date'].isnull().sum()}")
print(f"Zakres dat: {matches['Date'].min().date()} – {matches['Date'].max().date()}")

Nieprzetworzone daty: 0
Zakres dat: 2005-07-29 – 2025-05-25


## Uzupełnienie brakujących godzin meczów

Starsze sezony nie mają kolumny `Time`. Brakujące godziny uzupełniam typową porą rozgrywania meczów: 17:00 w weekend i 21:00 w dzień roboczy.

In [8]:
# Dzień tygodnia: 0 = poniedziałek, 6 = niedziela
matches['DayOfWeek'] = matches['Date'].dt.dayofweek

default_time = np.where(matches['DayOfWeek'] >= 5, '17:00', '21:00')
matches['Time'] = matches['Time'].fillna(pd.Series(default_time, index=matches.index))

# Ujednolicam format godziny do HH:MM:SS (wymagany przez API pogodowe)
match_time = pd.to_datetime(matches['Time'], format='%H:%M', errors='coerce')
matches['Time'] = match_time.dt.strftime('%H:%M:%S')
matches['Hour'] = match_time.dt.hour.astype('Int64')

print(f"Pozostałe braki w kolumnie 'Time': {matches['Time'].isnull().sum()}")
matches[['Date', 'DayOfWeek', 'Time']].sample(10, random_state=42)

Pozostałe braki w kolumnie 'Time': 0


,Date,DayOfWeek,Time
14283,2006-12-23,5,17:00:00
26766,2020-08-02,6,17:00:00
36104,2025-02-01,5,17:30:00
34757,2021-08-15,6,19:00:00
24969,2015-10-28,2,21:00:00
31392,2012-10-06,5,17:00:00
32551,2015-10-23,4,21:00:00
17039,2014-03-09,6,17:00:00
2799,2014-09-26,4,21:00:00
27486,2022-04-25,0,19:45:00


## Typy danych i kolejność kolumn

In [9]:
stats_cols = [
    'FTHG', 'FTAG', 'HTHG', 'HTAG', 'HS', 'AS', 'HST', 'AST',
    'HF', 'AF', 'HC', 'AC', 'HY', 'AY', 'HR', 'AR', 'DayOfWeek'
]
for col in stats_cols:
    matches[col] = pd.to_numeric(matches[col], errors='coerce').astype('Int64')

# Usuwam kolumny pomocnicze i puste kolumny 'Unnamed'
unnamed_cols = [col for col in matches.columns if 'Unnamed' in str(col)]
matches = matches.drop(columns=['Div', 'Team'] + unnamed_cols)

first_columns = [
    'Division', 'Season', 'HomeTeam', 'AwayTeam', 'City', 'Stadium',
    'Latitude', 'Longitude', 'Date', 'Time'
]
other_columns = [col for col in matches.columns if col not in first_columns]
matches = matches[first_columns + other_columns]

matches.head()

,Division,Season,HomeTeam,AwayTeam,City,Stadium,Latitude,Longitude,Date,Time,...,BFECH,BFECD,BFECA,BFEC>2.5,BFEC<2.5,BFECAHH,BFECAHA,Referee,DayOfWeek,Hour
0,Bundesliga,2005-06,Bayern Munich,M'gladbach,Monachium,Allianz Arena,48.2188,11.6247,2005-08-05,21:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,21
1,Bundesliga,2005-06,Duisburg,Stuttgart,Duisburg,Schauinsland-Reisen-Arena,51.4094,6.7831,2005-08-06,17:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,17
2,Bundesliga,2005-06,FC Koln,Mainz,Koln,RheinEnergieStadion,50.9334,6.8751,2005-08-06,17:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,17
3,Bundesliga,2005-06,Hamburg,Nurnberg,Hamburg,Volksparkstadion,53.5870,9.8976,2005-08-06,17:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,17
4,Bundesliga,2005-06,Hannover,Hertha,Hanower,Heinz von Heiden-Arena,52.3600,9.7303,2005-08-06,17:00:00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,17


In [10]:
matches.to_csv(PROCESSED_DIR / 'matches_clean.csv', index=False)
print(f'Zapisano {len(matches)} meczów do matches_clean.csv')

Zapisano 36271 meczów do matches_clean.csv
